In [9]:
from simple_pid import PID
import time


class Bear:     # This is the 'constructor'
    def __init__(self, name):
        self.name = name   # Attribute
        self.h = 0
        self.speed = 0
        self.m = 10

        # added lines
        self.target_h = 0
        self.max_power = 50
        self.min_power = -50
        self.max_speed = 20
        self.sun_level = 80
        self.cloud_level = 40

        # PID controller:
        # input  -> current height from sensor
        # output -> engine power
        self.pid = PID(1.4, 0.2, 0.1, setpoint=self.target_h)
        self.pid.output_limits = (self.min_power, self.max_power)
        self.pid.sample_time = None

    def singsong(self):
        return f"Up, Down, Touch the Ground"

    def calcPosition(self,curentH, curentSpeed, t):
        newH = 0
        newSpeed = 0

        if t <= 0:
            raise ValueError("t must be > 0")

        # sensor sends current height into PID
        engine_power = self.pid(curentH)

        # simple plant / engine model
        # acceleration = power / mass
        acceleration = engine_power / self.m

        newSpeed = curentSpeed + acceleration * t

        # limit speed
        if newSpeed > self.max_speed:
            newSpeed = self.max_speed
        if newSpeed < -self.max_speed:
            newSpeed = -self.max_speed

        # calculate new height
        newH = curentH + newSpeed * t

        # floor at zero height
        if newH < 0:
            newH = 0
            newSpeed = 0

        # save state into object
        self.h = newH
        self.speed = newSpeed

        return newH, newSpeed


# Creating an 'instance' (object) of the class
pooh = Bear("Pooh")


class SunCloudSignalSystem:
    def __init__(self, bear):
        self.bear = bear

    def set_mode_sun(self):
        self.bear.target_h = self.bear.sun_level
        self.bear.pid.setpoint = self.bear.target_h

    def set_mode_clouds(self):
        self.bear.target_h = self.bear.cloud_level
        self.bear.pid.setpoint = self.bear.target_h

    def sensor_height(self):
        # H from sensor
        return self.bear.h

    def get_signal(self):
        h = self.sensor_height()

        if h >= self.bear.sun_level - 5:
            return "SUN"
        elif h <= self.bear.cloud_level + 5:
            return "CLOUDS"
        else:
            return "BETWEEN SUN AND CLOUDS"

    def step(self, dt):
        current_h = self.sensor_height()
        current_speed = self.bear.speed

        # PID -> engine -> new height
        new_h, new_speed = self.bear.calcPosition(current_h, current_speed, dt)

        signal = self.get_signal()

        return {
            "target_h": self.bear.target_h,
            "current_h": round(new_h, 2),
            "speed": round(new_speed, 2),
            "signal": signal
        }


def print_state(step_num, state):
    print(
        f"step={step_num:02d} | "
        f"target={state['target_h']:.2f} | "
        f"height={state['current_h']:.2f} | "
        f"speed={state['speed']:.2f} | "
        f"signal={state['signal']}"
    )


if __name__ == "__main__":
    print(pooh.singsong())

    system = SunCloudSignalSystem(pooh)

    print("\n--- Move to CLOUDS zone ---")
    system.set_mode_clouds()
    for i in range(1, 16):
        state = system.step(0.5)
        print_state(i, state)
        time.sleep(0.1)

    print("\n--- Move to SUN zone ---")
    system.set_mode_sun()
    for i in range(16, 36):
        state = system.step(0.5)
        print_state(i, state)
        time.sleep(0.1)

Up, Down, Touch the Ground

--- Move to CLOUDS zone ---
step=01 | target=40.00 | height=1.25 | speed=2.50 | signal=CLOUDS
step=02 | target=40.00 | height=3.75 | speed=5.00 | signal=CLOUDS
step=03 | target=40.00 | height=7.49 | speed=7.49 | signal=CLOUDS
step=04 | target=40.00 | height=12.34 | speed=9.69 | signal=CLOUDS
step=05 | target=40.00 | height=18.10 | speed=11.52 | signal=CLOUDS
step=06 | target=40.00 | height=24.56 | speed=12.92 | signal=CLOUDS
step=07 | target=40.00 | height=31.49 | speed=13.86 | signal=CLOUDS
step=08 | target=40.00 | height=38.64 | speed=14.30 | signal=CLOUDS
step=09 | target=40.00 | height=45.76 | speed=14.24 | signal=BETWEEN SUN AND CLOUDS
step=10 | target=40.00 | height=52.59 | speed=13.66 | signal=BETWEEN SUN AND CLOUDS
step=11 | target=40.00 | height=58.89 | speed=12.61 | signal=BETWEEN SUN AND CLOUDS
step=12 | target=40.00 | height=64.45 | speed=11.12 | signal=BETWEEN SUN AND CLOUDS
step=13 | target=40.00 | height=69.08 | speed=9.25 | signal=BETWEEN SUN

In [7]:
import pytest

from Untitled1 import Bear, SunCloudSignalSystem


def test_bear_constructor():
    bear = Bear("Pooh")

    assert bear.name == "Pooh"
    assert bear.h == 0
    assert bear.speed == 0
    assert bear.m == 10


def test_singsong():
    bear = Bear("Pooh")
    assert bear.singsong() == "Up, Down, Touch the Ground"


def test_calc_position_raises_for_zero_time():
    bear = Bear("Pooh")

    with pytest.raises(ValueError):
        bear.calcPosition(0, 0, 0)


def test_calc_position_raises_for_negative_time():
    bear = Bear("Pooh")

    with pytest.raises(ValueError):
        bear.calcPosition(0, 0, -1)


def test_calc_position_returns_numbers():
    bear = Bear("Pooh")
    bear.pid.setpoint = 50

    new_h, new_speed = bear.calcPosition(0, 0, 1)

    assert isinstance(new_h, (int, float))
    assert isinstance(new_speed, (int, float))


def test_height_never_goes_below_zero():
    bear = Bear("Pooh")

    new_h, new_speed = bear.calcPosition(0, -10, 1)

    assert new_h >= 0
    assert new_speed >= 0


def test_system_set_mode_clouds():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    system.set_mode_clouds()

    assert bear.target_h == bear.cloud_level
    assert bear.pid.setpoint == bear.cloud_level


def test_system_set_mode_sun():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    system.set_mode_sun()

    assert bear.target_h == bear.sun_level
    assert bear.pid.setpoint == bear.sun_level


def test_signal_clouds():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    bear.h = 20
    assert system.get_signal() == "CLOUDS"


def test_signal_sun():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    bear.h = 90
    assert system.get_signal() == "SUN"


def test_signal_between():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    bear.h = 60
    assert system.get_signal() == "BETWEEN SUN AND CLOUDS"


def test_step_returns_expected_keys():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    system.set_mode_sun()
    state = system.step(0.5)

    assert "target_h" in state
    assert "current_h" in state
    assert "speed" in state
    assert "signal" in state


def test_pid_moves_bear_toward_cloud_target():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    system.set_mode_clouds()

    initial_distance = abs(bear.target_h - bear.h)

    for _ in range(10):
        system.step(0.5)

    final_distance = abs(bear.target_h - bear.h)

    assert final_distance < initial_distance


def test_pid_moves_bear_toward_sun_target():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    system.set_mode_sun()

    initial_distance = abs(bear.target_h - bear.h)

    for _ in range(15):
        system.step(0.5)

    final_distance = abs(bear.target_h - bear.h)

    assert final_distance < initial_distance


def test_speed_is_limited():
    bear = Bear("Pooh")
    system = SunCloudSignalSystem(bear)

    system.set_mode_sun()

    for _ in range(50):
        system.step(0.5)

    assert -bear.max_speed <= bear.speed <= bear.max_speed

ModuleNotFoundError: No module named 'Untitled1'